# 02. Customize Config And Rerun Selected Steps

Use this notebook when you already have a prepared `.h5ad` and want to tune analysis parameters or regenerate selected outputs.

In [ ]:
from pathlib import Path

from perturbflow.analyzer.config import PipelineConfig
from perturbflow import run_analysis

PROJECT = Path.cwd()
PREPARED_H5AD = PROJECT / "prepared" / "my_data.perturbflow.h5ad"
RESULTS_DIR = PROJECT / "results" / "my_first_run"
CUSTOM_CONFIG = PROJECT / "configs" / "my_custom_config.json"

RUN_ANALYSIS = False

## Create a custom config

The object below starts from package defaults and changes a few common knobs. Save the JSON and pass it to `perturbflow run` or `run_analysis()`.

In [ ]:
cfg = PipelineConfig()
cfg.min_cells_per_perturbation = 10
cfg.deg_n_top_perturbations = 10
cfg.genenet_n_top_genes = 80
cfg.genenet_corr_threshold = 0.25
cfg.bundle_top_de_per_pert = 300

CUSTOM_CONFIG.parent.mkdir(parents=True, exist_ok=True)
cfg.to_json(CUSTOM_CONFIG)
print(f"Wrote: {CUSTOM_CONFIG}")

## Rerun selected steps

Examples:

- `steps=["deg", "report", "bundle"]` updates DEG tables and reports.
- `force_steps=["report"]` rebuilds reports even if checkpointed.
- `clear_from="deg"` invalidates the checkpoint from DEG onward.

In [ ]:
if RUN_ANALYSIS:
    run_analysis(
        input_path=PREPARED_H5AD,
        output_dir=RESULTS_DIR,
        config_path=CUSTOM_CONFIG,
        steps=["deg", "report", "bundle"],
        resume=True,
        force_steps=["report"],
    )
else:
    print("Set RUN_ANALYSIS=True after checking PREPARED_H5AD and RESULTS_DIR.")

## Equivalent command line

```bash
perturbflow run \
  --input prepared/my_data.perturbflow.h5ad \
  --output results/my_first_run \
  --config configs/my_custom_config.json \
  --steps deg,report,bundle \
  --force-steps report \
  --resume
```